# Publishing the trained model to Hugging Face

This notebook copies the trained checkpoint from Google Drive to the Hugging Face model repository [`leeely/segformer-b2-turkey-building-damage`](https://huggingface.co/leeely/segformer-b2-turkey-building-damage), together with the model card and the licence. It only needs to be run once, by the repository owner. Readers who want the model do not need it; notebook 01 shows how to download the weights.

The checkpoint is the one written by notebook 01 during training: `best_model.pt`, the epoch with the lowest validation loss. Besides the weights it holds the optimiser and scheduler state. We drop those before uploading because they are only needed to resume training. The training and validation loss curves and the class weights used in the loss are kept.

## 1. Setup

In [ ]:
!pip install -q huggingface_hub

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_PATH = "/content/drive/MyDrive/segformer_checkpoints/best_model.pt"
REPO_ID = "leeely/segformer-b2-turkey-building-damage"
GITHUB_RAW = "https://raw.githubusercontent.com/leeelyehezkel/Turkey-Building-Data/main"
RELEASE_DIR = "/content/hf_release"

## 2. Log in

`notebook_login` opens a field for a Hugging Face access token with write permission (create one at https://huggingface.co/settings/tokens). The token is stored only in this Colab session.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Check the checkpoint and keep only what inference needs

The printed epoch and validation loss should be 21 and 1.5627, the values reported in notebook 01. If they differ, the file on Drive is not the checkpoint used in the paper.

In [ ]:
import os
import torch

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
print("Keys in checkpoint:", list(checkpoint.keys()))
print("Epoch:", checkpoint.get("epoch"))
print("Best validation loss:", checkpoint.get("best_val_loss"))

release = {
    "model_state_dict": checkpoint["model_state_dict"],
    "epoch": checkpoint.get("epoch"),
    "best_val_loss": checkpoint.get("best_val_loss"),
    "best_epoch": checkpoint.get("best_epoch"),
    "class_weights": checkpoint.get("class_weights"),
    "train_losses": checkpoint.get("train_losses"),
    "val_losses": checkpoint.get("val_losses"),
}

os.makedirs(RELEASE_DIR, exist_ok=True)
torch.save(release, os.path.join(RELEASE_DIR, "best_model.pt"))
print(f"Saved {os.path.getsize(os.path.join(RELEASE_DIR, 'best_model.pt')) / 1e6:.1f} MB")

## 4. Check that the weights load into the architecture

This rebuilds the model exactly as in notebook 01 (Cityscapes SegFormer-B2 with a new 9-class head) and loads the weights. `load_state_dict` fails loudly if any tensor is missing or has the wrong shape.

In [ ]:
!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp

model = smp.from_pretrained("smp-hub/segformer-b2-1024x1024-city-160k")
model.segmentation_head = torch.nn.Conv2d(
    model.segmentation_head[0].in_channels, 9, kernel_size=1
)
model.load_state_dict(release["model_state_dict"])
print("Weights load correctly.")

## 5. Add the model card and licence, and upload

The model card (`README.md`) and the NVIDIA SegFormer licence are kept in the GitHub repository under `huggingface/`, so the two repositories stay in sync. The licence must travel with the weights because they are a derivative of NVIDIA's SegFormer.

In [ ]:
import urllib.request
from huggingface_hub import HfApi

for name in ["README.md", "LICENSE"]:
    urllib.request.urlretrieve(f"{GITHUB_RAW}/huggingface/{name}", os.path.join(RELEASE_DIR, name))

api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=False, exist_ok=True)
api.upload_folder(folder_path=RELEASE_DIR, repo_id=REPO_ID, repo_type="model",
                  commit_message="Release SegFormer-B2 checkpoint (epoch 21)")
print(f"https://huggingface.co/{REPO_ID}")

## 6. Check the download

Downloading the file again from the Hub and comparing it with the local copy confirms the upload is complete.

In [ ]:
from huggingface_hub import hf_hub_download

downloaded = torch.load(hf_hub_download(REPO_ID, "best_model.pt", force_download=True), map_location="cpu")
same = all(torch.equal(downloaded["model_state_dict"][k], v) for k, v in release["model_state_dict"].items())
print("Downloaded weights identical to local copy:", same)